# 01 · Cómo funciona Laya

**Laya** no es un LLM. Un LLM es *autoregresivo*: predice un token, lo añade a la entrada,
vuelve a predecir. Para clasificar te devuelve texto que tienes que parsear.

Laya es un **encoder + cabeza de decisión**. Una sola pasada hacia adelante, y la salida son
números: probabilidades sobre las opciones que tú definas.

| | LLM | Laya |
|---|---|---|
| salida | tokens | probabilidades |
| pasadas | N (una por token) | 1 |
| parsear | sí | no |
| alucinar | puede | no hay nada que alucinar |
| latencia (CPU) | segundos | ~25 ms |

La contrapartida: **solo responde preguntas que le tipas tú**. No conversa, no resume,
no razona en cadena.

Variantes del modelo:

| checkpoint | backbone | params | contexto | para qué |
|---|---|---|---|---|
| `convaiinnovations/laya` | ModernBERT-large | 421M | 512 | inglés |
| `convaiinnovations/laya-multilingual` | mmBERT-base | 322M | 1024–8192 | 100+ idiomas, ~2.2x más rápido |
| `convaiinnovations/laya-typed-decisions` | ModernBERT-large | 421M | 1024 | typed-decisions (0.766 acc) |

In [ ]:
import json
import time

import pandas as pd
from laya import Router

t0 = time.perf_counter()
router = Router()  # descarga el checkpoint la primera vez (~1.7 GB)
print(f"Router creado en {time.perf_counter() - t0:.2f} s")
print("(los checkpoints se cargan de forma diferida, en la primera predict)")

## La primera llamada

Fíjate en dos cosas de la salida cruda: `usage.output_tokens` y `routing`.

In [ ]:
estado = (
    "Hi, we were billed twice for March. Please refund the duplicate today "
    "or we will cancel our plan."
)

r = router.predict(estado, {
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"}
})

print(json.dumps(r, indent=2, ensure_ascii=False, default=str))

`"output_tokens": 0`.

Ahí está todo. No hay decodificación, no hay bucle de generación. De ahí los ~25 ms.

`input_tokens` sí cuenta — e incluye **el texto más tus preguntas y criterios**.
Presupuesto de tokens: el checkpoint inglés tiene 512. Las descripciones se los comen.

## Los tres tipos de pregunta

| tipo | pregunta | devuelve |
|---|---|---|
| `choice` | ¿cuál de estas opciones? | la opción + probabilidad de cada una |
| `score` | ¿dónde en esta escala? | posición continua en tus niveles |
| `noul` | ¿es esto cierto? | la probabilidad de que lo sea |

Todas las preguntas de una llamada se responden **en una sola pasada**. Añadir preguntas
es casi gratis: aprovéchalo.

In [ ]:
PREGUNTAS = {
    # choice: criteria es un DICT etiqueta -> descripción.
    # Las descripciones NO son decorativas: son anclas de embedding.
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "other": "everything else",
        },
    },
    # score: criteria es una LISTA. El orden define la escala ordinal.
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["not urgent", "soon", "blocking"],
    },
    # noul: sin criteria. La pregunta ES la especificación.
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
}

r = router.predict(estado, PREGUNTAS)
a = r["answers"]

print("department :", a["department"]["choice"])
print("urgency    :", a["urgency"]["score"])
print("churn_risk :", a["churn_risk"]["noul"])
print("tokens in  :", r["usage"]["input_tokens"], "/ out:", r["usage"]["output_tokens"])

### `choice` — softmax sobre tus opciones

Las etiquetas (`billing`, `technical`...) **no estaban en el entrenamiento**. Se las pasas
en tiempo de ejecución con su descripción, y el modelo las compara contra el texto en el
mismo espacio de embeddings. Zero-shot: cambias el clasificador sin reentrenar nada.

In [ ]:
dep = a["department"]
pd.DataFrame(
    sorted(dep["probabilities"].items(), key=lambda kv: -kv[1]),
    columns=["opción", "probabilidad"],
).style.format({"probabilidad": "{:.4f}"}).hide(axis="index")

Ojo a `other: ~0.005`. **Casi nunca dispara.** `"everything else"` no es un concepto
que un encoder pueda embeber — es un agujero semántico. Si necesitas un cajón de sastre,
descríbelo con contenido real, o mejor: detéctalo con un umbral de confianza.

### `score` — no es un argmax, es la media esperada

`score` = suma de (índice × probabilidad). Verifícalo a mano:

In [ ]:
urg = a["urgency"]
esperado = sum(int(k) * v for k, v in urg["probabilities"].items())

print("probabilidades:", urg["probabilities"])
print("legend        :", urg["legend"])
print()
print(f"score devuelto : {urg['score']}")
print(f"media esperada : {esperado:.4f}")
print(f"argmax sería   : {max(urg['probabilities'], key=urg['probabilities'].get)}")
print()
print("-> coinciden: score es un valor CONTINUO, útil para ordenar o poner umbrales")

### `noul` — probabilidad cruda

Sin `criteria`. Redactar bien la pregunta es todo el trabajo — y es más sensible de lo
que parece (lo medimos en el notebook 02).

## Las dos confianzas

- **`answer_confidence`** = probabilidad de la opción ganadora. Cuánto pesa *esa* respuesta.
- **`confidence`** = certeza global de la distribución. Baja si la masa está repartida.

En `urgency` la `confidence` cae aunque `answer_confidence` sea alta: el modelo duda
entre dos niveles contiguos.

⚠️ La fórmula exacta de `confidence` no está documentada. Para lógica de producción
**calcula tú** lo que necesites sobre `probabilities` (margen entre las dos primeras,
entropía). Es robusto y no depende de una versión.

In [ ]:
filas = []
for nombre, resp in a.items():
    filas.append({
        "pregunta": nombre,
        "tipo": resp["type"],
        "confidence": resp.get("confidence"),
        "answer_confidence": resp.get("answer_confidence"),
        # la demo de HF devuelve rl_agent, laya 0.3.20 local devuelve action. No es estable.
        "act_prob": (resp.get("action") or resp.get("rl_agent", {})).get("act_probability"),
    })

pd.DataFrame(filas)

`act_probability` = "tengo suficiente para responder, no necesito más contexto".
Viene del entrenamiento **RLCD** (Reinforcement Learning for Calibrated Decisions):
el modelo aprende a calibrarse, no solo a acertar.

## El Router no es el modelo

`Router` es un **despachador**. Detecta script e idioma y elige checkpoint:

- latín + inglés → `laya` (512 tokens)
- cualquier otra cosa → `laya-multilingual` (hasta 8192 tokens, ~2.2x más rápido)

Si escribes en español, el `diacritic_rate` te manda al multilingüe. **Eso es una ventaja**:
16x más contexto para meter descripciones largas.

In [ ]:
textos = {
    "inglés": "The app crashes every time I open the settings page.",
    "español": "La aplicación se cierra cada vez que abro la configuración.",
    "hindi": "मुझसे मार्च में दो बार शुल्क लिया गया, कृपया डुप्लिकेट राशि वापस करें।",
}

filas = []
for etiqueta, texto in textos.items():
    r = router.predict(texto, {"department": PREGUNTAS["department"]})
    det = r["routing"]["detection"]
    filas.append({
        "idioma": etiqueta,
        "checkpoint": r["routing"]["model"],
        "motivo": r["routing"]["reason"],
        "script": det["script"],
        "diacríticos": det["diacritic_rate"],
        "no-latino": det["non_latin_fraction"],
        "elige": r["answers"]["department"]["choice"],
    })

pd.DataFrame(filas)

Puedes forzarlo y saltarte la detección — recomendable si sabes que tu entrada es
española y quieres los 8192 tokens siempre:

```python
router.predict(texto, preguntas, model="multilingual", max_len=8192)
```

O prescindir del Router del todo:

```python
import laya
agente = laya.load("convaiinnovations/laya")
agente_ml = laya.load("convaiinnovations/laya", subfolder="multilingual")
```

## Latencia real

La primera llamada incluye carga del modelo. Mide en caliente.

In [ ]:
una = {"churn_risk": PREGUNTAS["churn_risk"]}
router.predict(estado, una)  # calentamiento

for i in range(5):
    t = time.perf_counter()
    router.predict(estado, una)
    print(f"1 pregunta   run{i}: {(time.perf_counter() - t) * 1000:6.1f} ms")

router.predict(estado, PREGUNTAS)
for i in range(3):
    t = time.perf_counter()
    router.predict(estado, PREGUNTAS)
    print(f"3 preguntas  run{i}: {(time.perf_counter() - t) * 1000:6.1f} ms")

3 preguntas cuestan casi lo mismo que 1: **una sola pasada**. Por eso conviene
preguntar todo de golpe en vez de hacer N llamadas.

## Los tres límites que condicionan cualquier diseño

**1 · Calibración rota con 11+ opciones.** Al cargar salta:

```
RuntimeWarning: choice:11+ = 0.1005 -> 0.5
Treat confidence from the affected entries as uncalibrated.
```

El checkpoint trae temperaturas de calibración por tramo de cardinalidad, y la de 11+
viene fuera de rango. Con ≤10 opciones no te afecta. Con más, **la etiqueta sirve pero
el número no** — y cualquier diseño basado en umbrales se rompe.
→ Si necesitas más de 10 categorías: enruta en **dos niveles** (ámbito, luego detalle).

**2 · Presupuesto de tokens.** `input_tokens` = texto + preguntas + todas las
descripciones. 8 categorías a 60 tokens ya son ~500 → no cabe en 512.
→ Usa el checkpoint multilingüe.

**3 · Sin memoria.** Laya es **una pasada sin estado**. No aprende de decisiones
anteriores en inferencia. Nunca. Si quieres historial, va **en tu código**
(prior bayesiano sobre las probabilidades) o en un fine-tune.

Debilidades declaradas por el propio model card:
- `score` ordinales: flojos. Úsalos para **ordenar**, no para un `if`.
- sobreconfiado por defecto.
- el checkpoint base no vio datos de email.

Fuerte en: **routing**, clasificación, moderación, guardrails.

---
Siguiente: **02_experimentos.ipynb** — medir si estos números significan algo.